# Reprendre la vérification vLLM et API

Le test final du socle et du SFT est terminé et sauvegardé. Le premier essai vLLM a atteint le démarrage du moteur, puis l’API a échoué : le port 8080 était déjà occupé dans Colab. Je réserve désormais un port libre et le transmets à Uvicorn.

```text
poids SFT figés → vLLM → API sur port libre → trois cas fictifs + traces → sauvegarde → libération GPU
```

Je relance uniquement cette vérification technique. Les évaluations QA et triage restent inchangées. Une G4 ou une A100 suffit pour cet essai. Les journaux et le rapport sont copiés sur Drive avant la libération de la VM, même en cas d’échec. Cet essai ne constitue ni une évaluation clinique ni un hébergement public permanent.


In [ ]:
# Je reprends uniquement la vérification vLLM/API, sans relancer le test final.
from google.colab import drive, runtime
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess
import sys
import traceback
import zipfile

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/OCR_PROJET_14')
PAQUET = DRIVE / 'ocr14_verification_vllm_20261105.zip'
RACINE = Path('/content/OCR14_verification_vllm')
identifiant = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
SORTIES = DRIVE / 'evaluations' / f'vllm_reprise_{identifiant}'
SORTIES.mkdir(parents=True, exist_ok=True)
STATUT = {'test_final': 'conserve_sans_reexecution', 'vllm': 'non_execute'}

def executer(commande, nom):
    """Affiche l'étape et conserve le journal détaillé pour le diagnostic."""
    print('Étape :', nom, flush=True)
    journal = SORTIES / nom
    with journal.open('a', encoding='utf-8') as log:
        retour = subprocess.run(commande, cwd=RACINE, stdout=log, stderr=subprocess.STDOUT,
            env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    if retour:
        print(journal.read_text(errors='replace')[-7000:], flush=True)
        raise RuntimeError(f'Code {retour} ; journal : {nom}')

try:
    if hashlib.sha256(PAQUET.read_bytes()).hexdigest() != 'b167b3fbb77a333174f06b28082cad25ea27b4af888fc818bda11ee537f7aaf7':
        raise RuntimeError('Le paquet diffère de la version vérifiée.')
    RACINE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PAQUET) as archive:
        if not all((RACINE / n).resolve().is_relative_to(RACINE.resolve()) for n in archive.namelist()):
            raise ValueError('Chemin inattendu dans le paquet.')
        archive.extractall(RACINE)
    gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
    print('GPU attribué :', gpu.strip(), flush=True)
    (SORTIES / 'gpu.txt').write_text(gpu)
    source = DRIVE / 'experiences/sft_dpo_v3_20261105/sft_triage/adapter'
    adaptateur = RACINE / 'models/sft_triage_v3/adapter'
    shutil.copytree(source, adaptateur, dirs_exist_ok=True)
    executer([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], 'installation.log')
    environnement = '/content/ocr14_vllm_reprise_env'
    executer([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--seed',
              '--managed-python', environnement], 'installation.log')
    python_vllm = environnement + '/bin/python'
    executer([sys.executable, '-m', 'uv', 'pip', 'install', '--python', python_vllm,
              'vllm==0.32.0', 'uvicorn==0.54.0', 'jinja2==3.1.6', '--torch-backend=auto'], 'installation.log')
    executer([environnement + '/bin/vllm', 'download-kernels'], 'kernels.log')
    STATUT['vllm'] = 'en_cours'
    executer([python_vllm, '-m', 'serving.verifier_colab', '--adapter', str(adaptateur),
              '--output', str(SORTIES / 'vllm')], 'verification.log')
    STATUT['vllm'] = 'verifie_sur_trois_cas_fictifs'
    print((SORTIES / 'vllm/verification_endpoint.json').read_text(), flush=True)
except Exception:
    STATUT['vllm'] = 'echec'
    STATUT['erreur'] = traceback.format_exc()
    print(STATUT['erreur'], flush=True)
    journal_api = SORTIES / 'vllm/api.log'
    if journal_api.exists():
        print(journal_api.read_text(errors='replace')[-4000:], flush=True)
finally:
    (SORTIES / 'statut.json').write_text(json.dumps(STATUT, ensure_ascii=False, indent=2) + '\n')
    archive_locale = Path(f'/content/ocr14_vllm_reprise_{identifiant}.zip')
    with zipfile.ZipFile(archive_locale, 'w', zipfile.ZIP_DEFLATED) as archive:
        for fichier in sorted(SORTIES.rglob('*')):
            if fichier.is_file():
                archive.write(fichier, fichier.relative_to(SORTIES))
    archive_drive = DRIVE / archive_locale.name
    shutil.copy2(archive_locale, archive_drive)
    if hashlib.sha256(archive_drive.read_bytes()).digest() != hashlib.sha256(archive_locale.read_bytes()).digest():
        raise RuntimeError('Sauvegarde différente : la VM reste connectée.')
    print(json.dumps(STATUT, ensure_ascii=False, indent=2), flush=True)
    print('Archive :', archive_drive.name, flush=True)
    print('Synchronisation Drive, puis libération du GPU.', flush=True)
    drive.flush_and_unmount(timeout_ms=300_000)
    runtime.unassign()


{
  "date_utc": "2026-11-05T21:15:55.119125+00:00",
  "cas": "fictifs_sans_labels_cliniques",
  "mesures": [
    {
      "interaction_id": "585cba1d-eacd-4287-a0ee-eef75f120adc",
      "langue": "fr",
      "statut": "proposition_a_valider",
      "latence_http_ms": 6935.38,
      "latence_api_ms": 6880.41,
      "usage": {
        "prompt_tokens": 132,
        "total_tokens": 204,
        "completion_tokens": 72,
        "prompt_tokens_details": null,
        "completion_tokens_details": null
      },
      "modele": {
        "id": "ocr14-sft-triage-v3",
        "revision": "ea980cb0a6c2ae4b936e82123acc929f1cec04c1",
        "adapter_sha256": "7ca4d0257eb7cab0bccf9427efabb47f3a66bde3ec8bb9d76567f3cda086b7e5",
        "format_sft": "template_sans_reflexion_eos_natif_v2",
        "mlflow_run_id": "a63b597a3afd461588cddeb72375c856"
      }
    },
    {
      "interaction_id": "c425788a-9d51-48b1-a4d5-54d0b4374144",
      "langue": "en",
      "statut": "revue_humaine",
      "latence_ht

## Résultat conservé : bilan de la vérification réelle vLLM/API

Je vérifie le SFT retenu sur une A100 40 Go, avec vLLM 0.32.0. Le premier essai échoue car le port 8080 est déjà occupé. Je réserve ensuite un port libre transmis à Uvicorn ; un test CPU reproduit le conflit, puis le notebook 06 vérifie réellement l'inférence.

| Requête fictive | Statut | Temps HTTP |
|---|---|---:|
| FR, gorge | Proposition à valider | 6,935 s |
| EN, gorge | Revue humaine | 0,385 s |
| FR, poignet | Proposition à valider | 0,514 s |

La médiane est 0,514 s. Le premier appel peut inclure des initialisations ; trois requêtes séquentielles ne permettent pas de conclure sur la charge. Les trois traces SQLite sont concordantes. Les contrôles produisent 401 sans clé, 422 pour une entrée invalide et une question quand le cas est incomplet.

**Pourquoi la revue humaine ?** Le modèle écrit « delayed emergency » ; le contrat attend « delayed urgency ». Le niveau ESI seul ne suffit pas : l'API laisse la priorité à null. Je conserve cette limite observée sans modifier l'interprétation après l'essai. Une future normalisation doit être testée séparément sur des cas ambigus et contradictoires.

Les preuves sont dans `data/colab_archive/vllm_reprise_20261105_211011/`. La VM est temporaire et les appels internes : le Space public reste une démonstration Transformers, pas un endpoint vLLM permanent. Aucun résultat ne constitue une validation clinique.
